In [ ]:
# =========================================================
# FINAL SARIMAX EVALUATION (PANEL, WITH EXOGENOUS FEATURES)
# =========================================================

import numpy as np
import pandas as pd
import time
import os

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

# ---------------- CONFIG ----------------
DATA_PATH = "../../data/full_data.xlsx"

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

# TRAIN/TEST SPLIT
TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE   = pd.Timestamp("2022-03-31")  # training period end
TEST_START_DATE  = pd.Timestamp("2022-04-01")  # test period start

# === FILL THESE WITH YOUR BEST SARIMAX HYPERPARAMETERS FROM CV ===
# Example: (p,d,q) = (2,1,0), (P,D,Q,s) = (1,1,0,12)
BEST_ORDER          = (2, 1, 0)       # (p, d, q)
BEST_SEASONAL_ORDER = (1, 1, 0, 12)   # (P, D, Q, s)

# Exogenous features (same as other models)
continuous_cols = [
    "AverageNeighbourPrice","local_I","area_km2","centroid_x","centroid_y",
    "CoL_distance_km","LA_FE","sdlt_perc_threshold","dwelling_stock",
    "population","ashe_weekly","base_rate","claimant_count_prop",
    "planning_decisions_per_1000","planning_granted_prop",
    "rail_station_entry_exit","GDP","CPIH",
]

categorical_cols = [
    "LMIQuadrant__2","LMIQuadrant__3","LMIQuadrant__4",
    "Region_East of England","Region_London","Region_North East",
    "Region_North West","Region_South East","Region_South West",
    "Region_West Midlands","Region_Yorkshire and The Humber"
]

exog_cols = continuous_cols + categorical_cols

# =========================================================
# METRIC FUNCTIONS
# =========================================================
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(
        100.0 * np.mean(
            2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps)
        )
    )

def mase(y, yhat, y_train, m=12, eps=1e-8):
    """
    MASE using seasonal naive error with lag m on TRAIN period.
    y_train: all training targets (original scale).
    """
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)

    if len(y_train) <= m:
        return np.nan

    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return float(np.mean(np.abs(y - yhat)) / scale)

def directional_accuracy(df, entity_col, time_col, y_col, yhat_col):
    """
    Sign accuracy of month-on-month changes, averaged across LAs.
    """
    acc_list = []
    for la, sub in df.groupby(entity_col):
        sub = sub.sort_values(time_col)
        dy_true = sub[y_col].diff()
        dy_pred = sub[yhat_col].diff()
        mask = dy_true.notna() & dy_pred.notna() & (dy_true != 0)
        if mask.sum() == 0:
            continue
        correct = np.sign(dy_true[mask]) == np.sign(dy_pred[mask])
        acc_list.append(correct.mean())
    if not acc_list:
        return np.nan
    return float(np.mean(acc_list))

def morans_i(residuals, xs, ys, k=5):
    """
    Moran's I using k-NN weights on LA centroids.
    residuals: [N], xs, ys: [N]
    """
    residuals = np.asarray(residuals)
    xs = np.asarray(xs)
    ys = np.asarray(ys)
    N = len(residuals)
    coords = np.column_stack([xs, ys])

    nbrs = NearestNeighbors(n_neighbors=k+1).fit(coords)
    _, indices = nbrs.kneighbors(coords)

    W = np.zeros((N, N), dtype=float)
    for i in range(N):
        for j in indices[i, 1:]:
            W[i, j] = 1.0
            W[j, i] = 1.0

    S0 = W.sum()
    if S0 == 0:
        return np.nan

    x = residuals
    x_bar = x.mean()
    num = 0.0
    for i in range(N):
        for j in range(N):
            num += W[i, j] * (x[i] - x_bar) * (x[j] - x_bar)
    den = ((x - x_bar) ** 2).sum()
    if den == 0:
        return np.nan

    I = (N / S0) * (num / den)
    return float(I)

# =========================================================
# LOAD DATA & BUILD PANEL
# =========================================================
df_full = pd.read_excel(DATA_PATH, parse_dates=[TIME_COL])
df_full = df_full.sort_values([ENTITY_COL, TIME_COL]).reset_index(drop=True)

# robust centroids (for Moran's I)
df_full[["centroid_x", "centroid_y"]] = (
    df_full.groupby(ENTITY_COL)[["centroid_x", "centroid_y"]]
           .ffill()
           .bfill()
)

centroid_df = (
    df_full.drop_duplicates(ENTITY_COL)
           .set_index(ENTITY_COL)[["centroid_x", "centroid_y"]]
)

# restrict to >= TRAIN_START_DATE
df = df_full[df_full[TIME_COL] >= TRAIN_START_DATE].copy()
df = df.sort_values([TIME_COL, ENTITY_COL]).reset_index(drop=True)

la_order = sorted(df[ENTITY_COL].unique())
N = len(la_order)
print("Number of LAs used:", N)

centroid_df = centroid_df.loc[la_order]
bad_las = centroid_df[centroid_df.isna().any(axis=1)].index.tolist()
if bad_las:
    print(f"⚠ Dropping {len(bad_las)} LAs with missing centroids:", bad_las)
    centroid_df = centroid_df.dropna()
    la_order = centroid_df.index.tolist()
    df = df[df[ENTITY_COL].isin(la_order)].copy()
    N = len(la_order)
    print("Updated number of LAs:", N)

# Ensure one row per (Date, LA)
df = (
    df.sort_values([TIME_COL, ENTITY_COL])
      .drop_duplicates(subset=[TIME_COL, ENTITY_COL], keep="last")
      .copy()
)

dates = pd.Index(sorted(df[TIME_COL].unique()))
T_total = len(dates)
print("Total months in eval period:", T_total)

full_index = pd.MultiIndex.from_product(
    [dates, la_order],
    names=[TIME_COL, ENTITY_COL]
)

df_panel = (
    df.set_index([TIME_COL, ENTITY_COL])
      .reindex(full_index)
      .sort_index()
)

# ffill/bfill features + target within each LA
df_panel[exog_cols + [TARGET_COL]] = (
    df_panel[exog_cols + [TARGET_COL]]
        .groupby(level=ENTITY_COL)
        .ffill()
        .bfill()
)

missing_total = df_panel[exog_cols + [TARGET_COL]].isna().sum().sum()
if missing_total > 0:
    print(f"⚠ {missing_total} NaNs after ffill/bfill. Filling with column means.")
    col_means = df_panel[exog_cols + [TARGET_COL]].mean()
    df_panel[exog_cols + [TARGET_COL]] = df_panel[exog_cols + [TARGET_COL]].fillna(col_means)

print("NaNs after panel completion:",
      df_panel[exog_cols + [TARGET_COL]].isna().sum().sum())

# =========================================================
# TRAIN / TEST SPLIT
# =========================================================
train_mask = (dates >= TRAIN_START_DATE) & (dates <= TRAIN_END_DATE)
test_mask  = (dates >= TEST_START_DATE)

train_dates = dates[train_mask]
test_dates  = dates[test_mask]

print(f"Train: {train_dates[0].date()} → {train_dates[-1].date()}")
print(f"Test : {test_dates[0].date()} → {test_dates[-1].date()}")

df_train = df_panel.loc[(train_dates, slice(None)), :].copy()
df_test  = df_panel.loc[(test_dates,  slice(None)), :].copy()

# =========================================================
# LEAK-FREE SCALING OF EXOGENOUS FEATURES
# =========================================================
scaler = StandardScaler()
scaler.fit(df_train[continuous_cols])

df_train_scaled = df_train.copy()
df_test_scaled  = df_test.copy()

df_train_scaled.loc[:, continuous_cols] = scaler.transform(df_train[continuous_cols])
df_test_scaled.loc[:,  continuous_cols] = scaler.transform(df_test[continuous_cols])

# For MASE scaling: all training targets in original units (all LAs)
y_train_all = df_train[TARGET_COL].values

# =========================================================
# FIT SARIMAX PER LA, FORECAST TEST PERIOD
# =========================================================
print("\n=== Fitting SARIMAX per LA and forecasting test period ===")
start_time = time.time()

rows_true = []
rows_pred = []

for la in la_order:
    sub_train = df_train_scaled.xs(la, level=ENTITY_COL).sort_index()
    sub_test  = df_test_scaled.xs(la,  level=ENTITY_COL).sort_index()

    if sub_train[TARGET_COL].isna().all() or sub_test[TARGET_COL].isna().all():
        print(f"  LA {la}: missing train/test data; skipping.")
        continue

    endog_train = sub_train[TARGET_COL].values.astype(float)
    exog_train  = sub_train[exog_cols].values.astype(float)
    exog_test   = sub_test[exog_cols].values.astype(float)
    n_test      = len(sub_test)

    if n_test == 0:
        print(f"  LA {la}: no test data; skipping.")
        continue

    try:
        model = SARIMAX(
            endog=endog_train,
            exog=exog_train,
            order=BEST_ORDER,
            seasonal_order=BEST_SEASONAL_ORDER,
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        res = model.fit(disp=False)

        y_forecast = res.forecast(steps=n_test, exog=exog_test)
        y_true     = sub_test[TARGET_COL].values.astype(float)

        la_codes = np.array([la] * n_test)
        la_dates = sub_test.index.get_level_values(TIME_COL).to_numpy()

        rows_true.append(
            pd.DataFrame({
                TIME_COL: la_dates,
                ENTITY_COL: la_codes,
                "y_true": y_true
            })
        )
        rows_pred.append(
            pd.DataFrame({
                TIME_COL: la_dates,
                ENTITY_COL: la_codes,
                "y_pred": y_forecast.values
            })
        )

        print(f"  LA {la}: fitted successfully, forecast {n_test} months.")

    except Exception as e:
        print(f"  LA {la}: SARIMAX failed ({e}). Skipping this LA.")
        continue

train_time = time.time() - start_time
print(f"\nTotal SARIMAX fitting + forecasting time: {train_time:.1f} seconds")

if not rows_true:
    raise RuntimeError("No successful LA fits; cannot compute evaluation metrics.")

df_true = pd.concat(rows_true, axis=0, ignore_index=True)
df_pred = pd.concat(rows_pred, axis=0, ignore_index=True)

df_test_eval = (
    df_true.merge(df_pred, on=[TIME_COL, ENTITY_COL], how="inner")
            .sort_values([TIME_COL, ENTITY_COL])
            .reset_index(drop=True)
)

df_test_eval["resid"] = df_test_eval["y_true"] - df_test_eval["y_pred"]

# =========================================================
# GLOBAL METRICS
# =========================================================
global_mae   = mae(df_test_eval["y_true"], df_test_eval["y_pred"])
global_rmse  = rmse(df_test_eval["y_true"], df_test_eval["y_pred"])
global_smape = smape(df_test_eval["y_true"], df_test_eval["y_pred"])
global_mase  = mase(df_test_eval["y_true"], df_test_eval["y_pred"], y_train_all, m=12)

print("\n=== Global accuracy ===")
print(f"MAE   : {global_mae:,.3f}")
print(f"RMSE  : {global_rmse:,.3f}")
print(f"sMAPE : {global_smape:.3f}%")
print(f"MASE  : {global_mase:.3f}")

# =========================================================
# ACROSS-LA CONSISTENCY
# =========================================================
la_mae = (
    df_test_eval.groupby(ENTITY_COL)
                .apply(lambda g: mae(g["y_true"], g["y_pred"]))
)

median_mae = float(np.median(la_mae.values))
p75_mae    = float(np.percentile(la_mae.values, 75))

print("\n=== Across-LA consistency ===")
print(f"Median LA MAE       : {median_mae:,.3f}")
print(f"75th percentile MAE : {p75_mae:,.3f}")

# =========================================================
# SPATIO-TEMPORAL DIAGNOSTICS
# =========================================================
# Moran's I on mean residual per LA
la_resid_mean = df_test_eval.groupby(ENTITY_COL)["resid"].mean()
centroids = centroid_df.loc[la_resid_mean.index][["centroid_x", "centroid_y"]]

I_moran = morans_i(
    residuals=la_resid_mean.values,
    xs=centroids["centroid_x"].values,
    ys=centroids["centroid_y"].values,
    k=5
)

# Ljung–Box on mean residual over time
monthly_resid = (
    df_test_eval.groupby(TIME_COL)["resid"]
                .mean()
                .sort_index()
)

lb = acorr_ljungbox(monthly_resid, lags=[12], return_df=True)
lb_stat = float(lb["lb_stat"].iloc[0])
lb_p    = float(lb["lb_pvalue"].iloc[0])

print("\n=== Spatio-temporal diagnostics ===")
print(f"Moran's I (mean residuals across LAs): {I_moran:.4f}")
print(f"Ljung–Box Q(12): stat={lb_stat:.3f}, p={lb_p:.4f}")

# =========================================================
# DIRECTIONAL ACCURACY & GROWTH-RATE ERROR
# =========================================================
dir_acc = directional_accuracy(df_test_eval, ENTITY_COL, TIME_COL, "y_true", "y_pred")

# approximate 12-month growth-rate error
# build full [T, N] arrays for y_true & y_pred on test-support
dates_all = dates
la_array  = np.array(la_order)

# map (Date, LA) in df_test_eval back to indices
date_to_idx = {d: i for i, d in enumerate(dates_all)}

y_true_full = np.full((T_total, N), np.nan, dtype=float)
y_pred_full = np.full((T_total, N), np.nan, dtype=float)

for _, row in df_test_eval.iterrows():
    t = date_to_idx[row[TIME_COL]]
    n = np.where(la_array == row[ENTITY_COL])[0][0]
    y_true_full[t, n] = row["y_true"]
    y_pred_full[t, n] = row["y_pred"]

errs = []
test_start_idx = np.where(dates_all == TEST_START_DATE)[0][0]

for t in range(test_start_idx, T_total):
    t_prev = t - 12
    if t_prev < 0:
        continue
    true_t    = y_true_full[t]      # [N]
    true_prev = y_true_full[t_prev] # [N]
    pred_t    = y_pred_full[t]      # [N]
    mask = (~np.isnan(pred_t)) & (~np.isnan(true_t)) & (~np.isnan(true_prev)) & \
           (true_t > 0) & (true_prev > 0)
    if not mask.any():
        continue

    true_growth = np.log(true_t[mask]) - np.log(true_prev[mask])
    pred_growth = np.log(pred_t[mask]) - np.log(true_prev[mask])
    errs.append(np.abs(true_growth - pred_growth))

if errs:
    gre_mae = float(np.mean(np.concatenate(errs)))
else:
    gre_mae = np.nan

print("\n=== Direction & growth ===")
print(f"Directional accuracy (MoM sign)   : {dir_acc:.3f}")
print(f"Growth-rate error MAE (12-month)  : {gre_mae:.4f}")

# =========================================================
# SAVE SUMMARY METRICS TO CSV
# =========================================================
output_path = "../../results/sarimax_final_test_results.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

summary_df = pd.DataFrame([{
    "model_type": "SARIMAX_panel",
    "order": str(BEST_ORDER),
    "seasonal_order": str(BEST_SEASONAL_ORDER),

    # global metrics
    "MAE": global_mae,
    "RMSE": global_rmse,
    "sMAPE": global_smape,
    "MASE": global_mase,

    # across-LA
    "Median_LA_MAE": median_mae,
    "P75_LA_MAE": p75_mae,

    # spatio-temporal
    "Morans_I": I_moran,
    "LjungBox_Q12": lb_stat,
    "LjungBox_p": lb_p,

    # directional & growth
    "Directional_Accuracy": dir_acc,
    "GrowthRateError_MAE": gre_mae,

    # efficiency
    "Training_Time_sec": train_time,
    "Train_End_Date": TRAIN_END_DATE.strftime("%Y-%m-%d"),
    "Test_Start_Date": TEST_START_DATE.strftime("%Y-%m-%d"),
}])

summary_df.to_csv(output_path, index=False)
print("\n✅ Final SARIMAX evaluation metrics saved to:")
print(output_path)
